#  Arm Assembly Primer : `*p`, `p[i]` and `p->x`
# 1. Memory and addresses
* Data that do not fit in registers (arrays, structures, ...) are in <font color="blue">memory</font>
* Memory is a huge array of bytes; the position of a byte in it (0, 1, 2, ...) is its <font color="blue">address</font>
* A value larger than a byte (e.g., a `long`, 8 bytes) occupies consecutive bytes, and its address is that of its first byte
* <font color="blue">A pointer is just an address</font>, i.e., a 64-bit integer. Let's see it


In [ ]:
%%writefile addr.c
#include <stdio.h>

int main() {
  long a[4] = { 10, 20, 30, 40 };
  long * p = &a[0];
  printf("p     = %p (%ld)\n", (void *)p, (long)p);
  printf("p + 1 = %p (%ld)\n", (void *)(p + 1), (long)(p + 1));
  printf("&a[2] = %p (%ld)\n", (void *)&a[2], (long)&a[2]);
  printf("*(p + 2) = %ld, p[2] = %ld\n", *(p + 2), p[2]);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang -O1 addr.c -o addr
./addr

* `%p` prints a pointer in hexadecimal, and casting it to `long` shows the same number in decimal
* `p + 1` is larger than `p` by 8, not by 1: adding 1 to a pointer to `long` moves it to the next `long`, which is 8 bytes ahead
* `&a[2]` is `p + 2`, and `p[2]` is the same as `*(p + 2)`

# 2. Loads and stores

In [ ]:
%%writefile access.c
typedef struct {
  long x;
  long y;
  long z;
} point;

long deref(long * p) { return *p; }
long index_(long * p, long i) { return p[i]; }
long index3(long * p) { return p[3]; }
long member(point * q) { return q->y; }
long member_i(point * q, long i) { return q[i].z; }
int index_int(int * p, long i) { return p[i]; }
double index_double(double * p, long i) { return p[i]; }
void store(long * p, long i, long v) { p[i] = v; }

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang -O1 -S access.c
grep -vE '^\s+\.|^\s*//|^\.Lfunc' access.s

* You will find

<div>

| C | assembly | the address accessed |
|---|----------|----------------------|
| `*p` | `ldr x0, [x0]` | `p` |
| `p[i]` | `ldr x0, [x0, x1, lsl #3]` | `p + i * 8` |
| `p[3]` | `ldr x0, [x0, #24]` | `p + 24` |
| `q->y` | `ldr x0, [x0, #8]` | `q + 8` |
| `q[i].z` | `mov w8, #24` + `madd x8, x1, x8, x0` + `ldr x0, [x8, #16]` | `q + i * 24 + 16` |
| `p[i]` (`int`) | `ldr w0, [x0, x1, lsl #2]` | `p + i * 4` |
| `p[i]` (`double`) | `ldr d0, [x0, x1, lsl #3]` | `p + i * 8` |
| `p[i] = v` | `str x2, [x0, x1, lsl #3]` | `p + i * 8` |

</div>

* <font color="blue">`ldr`</font> (_load register_) reads a value from memory into a register; <font color="blue">`str`</font> (_store register_) writes a register into memory
* The part in `[...]` is the address
  * `[x0]` : the address in `x0`
  * `[x0, #24]` : `x0 + 24`
  * `[x0, x1, lsl #3]` : `x0 + (x1 << 3)` = `x0 + x1 * 8`
* The size of the value read or written is determined by the register: `x` = 8 bytes, `w` = 4 bytes, `d` = 8 bytes (`double`)

# 3. They are all the same thing
* `*p`, `p[i]` and `q->y` look different in C, but they all become "compute an address, then load (or store)"
  * `*p` : the address is `p` itself
  * `p[i]` : the address is `p + i * (the size of an element)`; in C, `p[i]` is _defined_ as `*(p + i)`
  * `q->y` : the address is `q + (the position of `y` in the structure)`; the members of `point` are laid out in order, `x` at 0, `y` at 8, `z` at 16, so a `point` occupies 24 bytes
  * `q[i].z` : the two combined, `q + i * 24 + 16`
* The compiler knows the sizes of types and the layouts of structures, and turns them into numbers (8, 24, 16, ...) in the instructions; nothing about types remains in the assembly code
* The CPU does not know whether a register holds a pointer or an integer; it is just a 64-bit number, which becomes an address when used in `[...]`

# 4. Summary
* memory is an array of bytes, and an address is a position in it
* a pointer is an address (a 64-bit integer)
* `*p`, `p[i]`, `p->x` all become `ldr`/`str` with an address of the form "base + constant" or "base + index × element size"